# SQL-запросы и дашборд

**Часть 4.** В предыдущих частях мы считали всё на Python. Здесь те же результаты получаем **средствами SQL** и собираем из них дашборд. Так работают аналитики в BI-инструментах вроде Redash: запрос возвращает таблицу, а график строится по ней.

Что внутри:
1. Загрузка данных в SQL-базу и представление с сегментами
2. Проверка качества данных запросом
3. Пять аналитических запросов: итоги по группам, воронка, эффект по сегментам, симуляция прибыли
4. Дашборд с интерактивными графиками (Plotly)

Тексты запросов лежат отдельно в папке `sql/` репозитория. Их можно вставить в Redash как есть: параметр `{{ cost }}` в запросе прибыли в Redash становится полем ввода.

В данных нет дат, поэтому динамики по дням здесь нет: показываем сравнение групп и сегментов.

In [17]:
import glob
import sqlite3
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
pd.set_option("display.width", 200)

paths = glob.glob("/kaggle/input/**/*Hillstrom*.csv", recursive=True) \
     or glob.glob("/kaggle/input/**/*.csv", recursive=True) \
     or glob.glob("../archive/*.csv")
raw = pd.read_csv(paths[0])
print("Файл:", paths[0], raw.shape)

# База в памяти. На Kaggle работает без установки: SQLite входит в Python
con = sqlite3.connect(":memory:")
raw.to_sql("email_campaign", con, index=False)

def q(sql, **params):
    """Выполняет запрос и подставляет параметры вида {{ name }}, как в Redash."""
    for k, v in params.items():
        sql = sql.replace("{{ " + k + " }}", str(v))
    return pd.read_sql(sql, con)

Файл: /kaggle/input/datasets/sumittarya/hillstrom-email-marketing/Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv (64000, 12)


## 1. Представление с сегментами

Исходная таблица `email_campaign` содержит сырые колонки. Поверх неё создаём представление `customers`, где уже есть удобные сегменты: новичок или нет, группа по давности, тип покупателя. Все следующие запросы работают с этим представлением.

In [18]:
SETUP = """
-- Таблица email_campaign загружается из CSV (Hillstrom Email Marketing).
-- Представление customers добавляет понятные сегменты.
DROP VIEW IF EXISTS customers;

CREATE VIEW customers AS
SELECT
    segment,
    recency,
    history,
    history_segment,
    zip_code,
    channel,
    newbie,
    mens,
    womens,
    CASE WHEN newbie = 1 THEN 'new' ELSE 'existing' END AS newbie_label,
    CASE
        WHEN recency <= 3 THEN '1-3'
        WHEN recency <= 6 THEN '4-6'
        WHEN recency <= 9 THEN '7-9'
        ELSE '10-12'
    END AS recency_bucket,
    CASE
        WHEN mens = 1 AND womens = 0 THEN 'only_mens'
        WHEN mens = 0 AND womens = 1 THEN 'only_womens'
        WHEN mens = 1 AND womens = 1 THEN 'both'
        ELSE 'neither'
    END AS buyer_type,
    visit,
    conversion,
    spend
FROM email_campaign;
"""

con.executescript(SETUP)
q("SELECT * FROM customers LIMIT 5")

,segment,recency,history,history_segment,zip_code,channel,newbie,mens,womens,newbie_label,recency_bucket,buyer_type,visit,conversion,spend
0,Womens E-Mail,10,142.440,2) $100 - $200,Surburban,Phone,0,1,0,existing,10-12,only_mens,0,0,0.000
1,No E-Mail,6,329.080,3) $200 - $350,Rural,Web,1,1,1,new,4-6,both,0,0,0.000
2,Womens E-Mail,7,180.650,2) $100 - $200,Surburban,Web,1,0,1,new,7-9,only_womens,0,0,0.000
3,Mens E-Mail,9,675.830,5) $500 - $750,Rural,Web,1,1,0,new,7-9,only_mens,0,0,0.000
4,Womens E-Mail,2,45.340,1) $0 - $100,Urban,Web,0,1,0,existing,1-3,only_mens,0,0,0.000


## 2. Проверка качества данных

Каждый показатель, кроме общего числа строк, должен быть нулём.

In [19]:
Q_QUALITY = """
-- Проверка качества данных: каждое значение, кроме rows_total, должно быть 0
SELECT
    COUNT(*) AS rows_total,
    SUM(CASE WHEN recency IS NULL OR history IS NULL OR segment IS NULL THEN 1 ELSE 0 END) AS rows_with_nulls,
    SUM(CASE WHEN conversion = 1 AND visit = 0 THEN 1 ELSE 0 END) AS purchase_without_visit,
    SUM(CASE WHEN spend > 0 AND conversion = 0 THEN 1 ELSE 0 END) AS spend_without_purchase,
    SUM(CASE WHEN spend < 0 THEN 1 ELSE 0 END) AS negative_spend,
    SUM(CASE WHEN mens = 0 AND womens = 0 THEN 1 ELSE 0 END) AS neither_mens_nor_womens
FROM customers;
"""

q(Q_QUALITY)

,rows_total,rows_with_nulls,purchase_without_visit,spend_without_purchase,negative_spend,neither_mens_nor_womens
0,64000,0,0,0,0,0


## 3. Итоги теста по группам

In [20]:
Q_SUMMARY = """
-- Итоги теста по группам: размер, визиты, конверсия, траты
SELECT
    segment AS grp,
    COUNT(*) AS customers,
    ROUND(100.0 * AVG(visit), 2) AS visit_rate_pct,
    ROUND(100.0 * AVG(conversion), 3) AS conversion_rate_pct,
    ROUND(AVG(spend), 3) AS spend_per_customer,
    ROUND(SUM(spend) * 1.0 / NULLIF(SUM(conversion), 0), 2) AS avg_order_value
FROM customers
GROUP BY segment
ORDER BY CASE segment WHEN 'No E-Mail' THEN 0 WHEN 'Mens E-Mail' THEN 1 ELSE 2 END;
"""

summary = q(Q_SUMMARY)
summary

,grp,customers,visit_rate_pct,conversion_rate_pct,spend_per_customer,avg_order_value
0,No E-Mail,21306,10.620,0.573,0.653,114.000
1,Mens E-Mail,21307,18.280,1.253,1.423,113.530
2,Womens E-Mail,21387,15.140,0.884,1.077,121.890


Цифры совпадают с первой частью: конверсия 0,57% в контроле против 1,25% у мужского и 0,88% у женского письма.

## 4. Воронка

In [21]:
Q_FUNNEL = """
-- Воронка по группам: получили письмо -> зашли на сайт -> купили
SELECT
    segment AS grp,
    COUNT(*) AS sent,
    SUM(visit) AS visited,
    SUM(conversion) AS purchased,
    ROUND(100.0 * SUM(visit) / COUNT(*), 2) AS sent_to_visit_pct,
    ROUND(100.0 * SUM(conversion) / NULLIF(SUM(visit), 0), 2) AS visit_to_purchase_pct,
    ROUND(100.0 * SUM(conversion) / COUNT(*), 3) AS sent_to_purchase_pct
FROM customers
GROUP BY segment
ORDER BY CASE segment WHEN 'No E-Mail' THEN 0 WHEN 'Mens E-Mail' THEN 1 ELSE 2 END;
"""

funnel = q(Q_FUNNEL)
funnel

,grp,sent,visited,purchased,sent_to_visit_pct,visit_to_purchase_pct,sent_to_purchase_pct
0,No E-Mail,21306,2262,122,10.620,5.390,0.573
1,Mens E-Mail,21307,3894,267,18.280,6.860,1.253
2,Womens E-Mail,21387,3238,189,15.140,5.840,0.884


Письмо повышает оба шага воронки: и долю зашедших на сайт, и долю купивших среди зашедших. Мужское письмо сильнее на обоих шагах.

## 5. Эффект письма по сегментам

In [22]:
Q_SEGMENTS = """
-- Эффект каждого письма относительно контроля в каждом сегменте
WITH long AS (
    SELECT 'newbie_label' AS dimension, newbie_label AS value, segment, visit, conversion, spend FROM customers
    UNION ALL
    SELECT 'channel', channel, segment, visit, conversion, spend FROM customers
    UNION ALL
    SELECT 'zip_code', zip_code, segment, visit, conversion, spend FROM customers
    UNION ALL
    SELECT 'history_segment', history_segment, segment, visit, conversion, spend FROM customers
    UNION ALL
    SELECT 'recency_bucket', recency_bucket, segment, visit, conversion, spend FROM customers
    UNION ALL
    SELECT 'buyer_type', buyer_type, segment, visit, conversion, spend FROM customers
),
agg AS (
    SELECT
        dimension, value, segment,
        COUNT(*) AS n,
        AVG(visit) AS visit_rate,
        AVG(conversion) AS conv_rate,
        AVG(spend) AS spend_avg
    FROM long
    GROUP BY dimension, value, segment
)
SELECT
    t.dimension,
    t.value,
    t.segment AS email,
    t.n AS n_email,
    c.n AS n_control,
    ROUND(100.0 * c.visit_rate, 2) AS control_visit_pct,
    ROUND(100.0 * t.visit_rate, 2) AS email_visit_pct,
    ROUND(100.0 * (t.visit_rate - c.visit_rate), 2) AS visit_lift_pp,
    ROUND(100.0 * (t.conv_rate - c.conv_rate), 3) AS conversion_lift_pp,
    ROUND(t.spend_avg - c.spend_avg, 3) AS spend_lift
FROM agg t
JOIN agg c
    ON c.dimension = t.dimension AND c.value = t.value AND c.segment = 'No E-Mail'
WHERE t.segment <> 'No E-Mail'
ORDER BY t.dimension, t.value, t.segment;
"""

segments = q(Q_SEGMENTS)
segments[segments.email == "Womens E-Mail"]

,dimension,value,email,n_email,n_control,control_visit_pct,email_visit_pct,visit_lift_pp,conversion_lift_pp,spend_lift
1,buyer_type,both,Womens E-Mail,2118,2149,18.050,25.170,7.110,0.533,0.162
3,buyer_type,only_mens,Womens E-Mail,9622,9638,9.980,11.090,1.110,0.063,0.278
5,buyer_type,only_womens,Womens E-Mail,9647,9519,9.580,16.980,7.400,0.511,0.632
7,channel,Multichannel,Womens E-Mail,2579,2606,12.850,17.560,4.710,0.705,1.158
9,channel,Phone,Womens E-Mail,9454,9327,8.720,13.180,4.460,0.173,0.232
11,channel,Web,Womens E-Mail,9354,9373,11.890,16.450,4.570,0.343,0.416
13,history_segment,1) $0 - $100,Womens E-Mail,7634,7612,8.490,13.160,4.680,0.379,0.626
15,history_segment,2) $100 - $200,Womens E-Mail,4727,4836,9.100,13.350,4.250,0.369,0.458
17,history_segment,3) $200 - $350,Womens E-Mail,4155,4044,12.640,16.250,3.610,-0.067,-0.323
19,history_segment,4) $350 - $500,Womens E-Mail,2188,2124,15.020,19.060,4.040,-0.216,-0.101


Для женского письма видно то же, что мы нашли в сегментном анализе: сильный эффект у покупавших женское (`only_womens`, `both`) и слабый у покупавших только мужское.

## 6. Симуляция прибыли
Запрос принимает параметр `cost` (стоимость одного письма) и показывает вклад каждой группы клиентов в прибыль на 1000 клиентов базы.

In [23]:
Q_PROFIT = """
-- Прибыль на 1000 клиентов базы от женского письма при стоимости письма {{ cost }} $
-- Две строки: покупавшие женское и остальные.
-- Сумма строк = слать всем; строка bought_womens = 1 = правило "слать только покупавшим женское".
WITH base AS (
    SELECT
        womens AS bought_womens,
        COUNT(*) AS n,
        AVG(CASE WHEN segment = 'Womens E-Mail' THEN spend END) AS spend_email,
        AVG(CASE WHEN segment = 'No E-Mail' THEN spend END) AS spend_control
    FROM customers
    WHERE segment IN ('Womens E-Mail', 'No E-Mail')
    GROUP BY womens
)
SELECT
    bought_womens,
    n AS customers,
    ROUND(100.0 * n / (SELECT SUM(n) FROM base), 1) AS share_of_base_pct,
    ROUND(spend_email - spend_control, 3) AS spend_lift,
    ROUND(1000.0 * n / (SELECT SUM(n) FROM base) * (spend_email - spend_control - {{ cost }}), 1) AS profit_per_1000_if_sent
FROM base
ORDER BY bought_womens;
"""

q(Q_PROFIT, cost=0.10)

,bought_womens,customers,share_of_base_pct,spend_lift,profit_per_1000_if_sent
0,0,19260,45.100,0.278,80.500
1,1,23433,54.900,0.544,243.800


Как читать: сумма двух строк равна прибыли от рассылки всем, а строка `bought_womens = 1` равна прибыли от рассылки только покупавшим женское.

In [24]:
rows = []
for cost in np.round(np.arange(0.0, 0.51, 0.01), 2):
    p = q(Q_PROFIT, cost=cost).set_index("bought_womens")["profit_per_1000_if_sent"]
    rows.append({"cost": cost, "Слать всем": p.sum(), "Слать только покупавшим женское": p.loc[1]})
profit = pd.DataFrame(rows)

# Стоимость письма, при которой правило начинает выигрывать у рассылки всем
better = profit[profit["Слать только покупавшим женское"] > profit["Слать всем"]]
breakeven = better["cost"].min() if len(better) else None
print("Правило выигрывает у рассылки всем начиная со стоимости письма:", breakeven)
profit.iloc[::10]

Правило выигрывает у рассылки всем начиная со стоимости письма: 0.28


,cost,Слать всем,Слать только покупавшим женское
0,0.000,424.300,298.700
10,0.100,324.300,243.800
20,0.200,224.200,188.900
30,0.300,124.200,134.000
40,0.400,24.300,79.200
50,0.500,-75.700,24.300


## 7. Дашборд

Три графика с интерактивностью: наведите курсор на столбцы, чтобы увидеть значения; на втором графике выпадающий список переключает признак сегментации. В конце ноутбук сохраняет всё в один файл `dashboard.html`.

In [25]:
COLORS = {"No E-Mail": "#9a9a9a", "Mens E-Mail": "#4c78a8", "Womens E-Mail": "#e45756"}
GROUPS = ["No E-Mail", "Mens E-Mail", "Womens E-Mail"]

# График 1. Воронка по группам
fig1 = make_subplots(rows=1, cols=3, subplot_titles=("Получили письмо -> зашли, %", "Зашли -> купили, %", "Получили письмо -> купили, %"))
f = funnel.set_index("grp").loc[GROUPS]
for i, col in enumerate(["sent_to_visit_pct", "visit_to_purchase_pct", "sent_to_purchase_pct"], start=1):
    fig1.add_trace(go.Bar(x=f.index, y=f[col], marker_color=[COLORS[g] for g in f.index],
                          text=f[col], textposition="outside", showlegend=False), row=1, col=i)
fig1.update_layout(title="Воронка по группам теста", height=420, template="plotly_white")
fig1.show()

In [26]:
# График 2. Эффект по сегментам с переключателем признака
DIM_LABELS = {
    "buyer_type": "Что покупал", "newbie_label": "Новичок или нет", "channel": "Канал",
    "zip_code": "Тип района", "history_segment": "Сумма прошлых покупок", "recency_bucket": "Давность покупки, мес.",
}
ORDER = {"recency_bucket": ["1-3", "4-6", "7-9", "10-12"]}
EMAILS = ["Mens E-Mail", "Womens E-Mail"]

fig2 = make_subplots(rows=2, cols=1, shared_xaxes=False, vertical_spacing=0.18,
                     subplot_titles=("Прирост доли визитов, п.п.", "Прирост конверсии, п.п."))
trace_dim = []
for di, (dim, label) in enumerate(DIM_LABELS.items()):
    d = segments[segments.dimension == dim].copy()
    if dim in ORDER:
        d["value"] = pd.Categorical(d["value"], ORDER[dim], ordered=True)
        d = d.sort_values(["value", "email"])
    for metric, row in (("visit_lift_pp", 1), ("conversion_lift_pp", 2)):
        for em in EMAILS:
            s = d[d.email == em]
            fig2.add_trace(go.Bar(x=s["value"].astype(str), y=s[metric], name=em, marker_color=COLORS[em],
                                  legendgroup=em, showlegend=(row == 1 and di == 0), visible=(di == 0)), row=row, col=1)
            trace_dim.append(dim)

buttons = []
for dim, label in DIM_LABELS.items():
    buttons.append(dict(label=label, method="update",
                        args=[{"visible": [t == dim for t in trace_dim]}]))
fig2.update_layout(title="Эффект письма по сегментам (разница с контролем)", height=620, template="plotly_white",
                   barmode="group", updatemenus=[dict(buttons=buttons, x=0.0, y=1.18, xanchor="left")])
fig2.show()

In [27]:
# График 3. Когда окупается таргетинг: прибыль при разной стоимости письма
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=profit["cost"], y=profit["Слать всем"], name="Слать всем", line=dict(color="#4c78a8", width=3)))
fig3.add_trace(go.Scatter(x=profit["cost"], y=profit["Слать только покупавшим женское"],
                          name="Слать только покупавшим женское", line=dict(color="#e45756", width=3)))
if breakeven is not None:
    fig3.add_vline(x=breakeven, line_dash="dash", line_color="gray",
                   annotation_text=f"правило выигрывает с ${breakeven:.2f}", annotation_position="top")
fig3.update_layout(title="Женское письмо: прибыль на 1000 клиентов базы", height=450, template="plotly_white",
                   xaxis_title="Стоимость одного письма, $", yaxis_title="Прибыль на 1000 клиентов, $")
fig3.show()

In [28]:
# Сохраняем все три графика в один HTML-файл
html = ["<html><head><meta charset='utf-8'><title>Email campaign dashboard</title></head><body>"]
for i, fig in enumerate([fig1, fig2, fig3]):
    html.append(fig.to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False))
html.append("</body></html>")
with open("dashboard.html", "w", encoding="utf-8") as fh:
    fh.write("\n".join(html))
print("Сохранено: dashboard.html")

Сохранено: dashboard.html


## Выводы

**Данные чистые.** Запрос проверки качества вернул нули по всем показателям: нет пропусков, покупок без визита, трат без покупки и отрицательных трат.

**SQL воспроизводит результаты Python-анализа.** Конверсия 0,57% в контроле, 1,25% у мужского и 0,88% у женского письма, траты на клиента $0,65, $1,42 и $1,08. Совпадение подтверждает, что расчёты в разных частях проекта согласованы.

**Воронка.** Письмо улучшает оба шага:

| Группа | Получили -> зашли | Зашли -> купили |
|---|---|---|
| Без письма | 10,6% | 5,4% |
| Мужское письмо | 18,3% | 6,9% |
| Женское письмо | 15,1% | 5,8% |

Основной прирост даёт первый шаг: письмо возвращает людей на сайт. Доля купивших среди зашедших растёт слабее.

**Сегменты.** Запрос эффекта по сегментам показывает то же, что мы нашли раньше: женское письмо сильно повышает визиты у покупавших женское (+7,1 и +7,4 п.п. для групп `both` и `only_womens`) и слабо у покупавших только мужское (+1,1 п.п.). Конверсия в последней группе почти не меняется (+0,06 п.п.).

**Деньги.** При стоимости письма $0,10 прибыль на 1000 клиентов базы составляет $324 при рассылке всем и $244 при рассылке только покупавшим женское. Правило «слать только покупавшим женское» начинает выигрывать у рассылки всем при стоимости письма от $0,28. Оценка упрощённая: правило подобрано по тем же данным, а не проверено на отложенной выборке, поэтому порог ориентировочный. Он согласуется с результатом uplift-модели (около $0,30).

**Что можно перенести в Redash.** Все запросы лежат в папке `sql/` и написаны так, чтобы работать в BI-инструменте без изменений. Запрос прибыли принимает параметр `{{ cost }}`, который в Redash превращается в поле ввода: аналитик или менеджер может менять стоимость письма и сразу видеть, какая стратегия выгоднее.

**Ограничения.** В данных нет дат, поэтому нет динамики по дням. Дашборд сравнивает группы и сегменты на накопленных за две недели результатах.